# Task 6: Offline VW Training — Production Fixed v2 (Python API)

## All Bugs Fixed in This Version

### BUG 1 (ROOT CAUSE — CRITICAL): `vw.learn(lines)` — Wrong input type
The original `train_candidate()` splits a block into `lines` (list[str]) and calls `vw.learn(lines)`.
For `--cb_adf`, the Python API requires the **full multiline block as a single string** OR a
list of pre-parsed `Example` objects. Passing `lines` (list[str]) raises
`RuntimeError: cb_adf: badly formatted example, only one line can have a cost`.
**Fix:** pass `block` (the full string, newline-joined) to `vw.learn()`.

### BUG 2 (CRITICAL): `vw.predict(stripped)` — Wrong input for multiline
Same issue in `evaluate_candidate()`: `strip_labels_from_block` returns `list[str]`,
but `vw.predict()` for a multiline `--cb_adf` model requires the full block string.
`predict()` also returns a **list of (action_index, probability) tuples** not a list of floats;
using `np.argmax` on tuples is wrong.
**Fix:** join stripped lines back to a string; extract action index from the tuple output.

### BUG 3 (CRITICAL): LABEL_RE mismatch between Task 5 output and Task 6 parser
Task 5 writes `0:{cost}:{prob:.6f} |action ...` (with the `0:` action-index prefix for cb_adf).
Task 6 FIXED regex strips that prefix: `r'^\s*(?P<cost>...):(?P<prob>...)\s+|action'`.
This means the regex fails to match Task 5 output and `parse_block_for_eval` returns None values,
crashing evaluation.
**Fix:** Accept both formats — `(?:0:)?` makes the action index optional.

### BUG 4: `--cb_adf` requires `--cb_explore_adf` for proper exploration during training
Using bare `--cb_adf` without an exploration strategy means DR/MTR types behave identically
and no diversity is injected. The sweep over `cb_type` in `('dr', 'mtr')` is only meaningful
with `--cb_explore_adf`. `--cb_adf` is the offline/OPE mode.
**Fix:** Use `--cb_explore_adf --epsilon 0.0` for offline replay (pure exploitation at train time),
while `--cb_type` still controls the loss estimator.

### BUG 5: `get_sum_loss() / get_weighted_examples()` returns 0.0 silently for cb_adf
VW's internal loss for cb_adf is always 0 via Python API because cb_adf loss is computed
per-label not per-example in the Python bindings. The fallback `np.mean(costs[:500])` is
correct in intent but samples only 500 blocks which biases the estimate.
**Fix:** Compute mean cost over all blocks for train_loss proxy.

### All other original fixes (FIX 3-5 from v1) are preserved:
- Quadratic interactions: `-q eu -q ea -q ua`
- `--clip_p` passed to VW args
- `--power_t 0` for fixed LR


In [ ]:
from __future__ import annotations

import itertools
import json
import re
import shutil
import os
import random
try:
    import boto3
except Exception:
    boto3 = None
import pandas as pd
import numpy as np

from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import List

from scipy.stats import entropy as scipy_entropy
try:
    import vowpalwabbit
except Exception:
    vowpalwabbit = None

print('Task 6 (FIXED v2) training setup loaded.')
if vowpalwabbit is not None:
    vw_version = getattr(vowpalwabbit, '__version__', 'unknown')
    print(f'VowpalWabbit Python package version: {vw_version}')
else:
    vw_cli = shutil.which('vw')
    if vw_cli:
        print(f'VowpalWabbit Python package not found; using CLI vw at {vw_cli}.')
    else:
        print('VowpalWabbit Python package and CLI vw were not found. Install one before training.')


def get_use_case_id(default: str = 'pl-aip-uplift') -> str:
    value = os.getenv('USE_CASE_ID', default).strip()
    if not value:
        raise ValueError('USE_CASE_ID must be non-empty.')
    return value


USE_CASE_ID = get_use_case_id('pl-aip-uplift')

# BUG 3 FIX: Accept BOTH formats from Task 5:
#   Format A (Task 5 original): '0:{cost}:{prob} |action ...'
#   Format B (Task 5 fixed):    '{cost}:{prob} |action ...'
# The (?:0:)? makes the action-index prefix optional.
LABEL_RE = re.compile(
    r'^\s*(?:0:)?(?P<cost>[0-9]*\.?[0-9]+):(?P<prob>[0-9]*\.?[0-9]+)\s+\|action\b'
)


@dataclass
class TrainingConfig:
    input_path:             Path
    output_dir:             Path
    s3_bucket:              str
    s3_input_key:           str
    s3_output_prefix:       str
    propensity_audit_key:   str
    training_intent:        str
    alignment_key:          str | None = None
    vw_binary:              str | None = None
    cleanup_local:          bool  = True
    random_seed:            int   = 42
    train_ratio:            float = 0.70
    valid_ratio:            float = 0.15
    min_actions_per_slate:  int   = 50
    l2_regularization:      float = 1e-6
    clip_p:                 float = 0.02
    quadratic_interactions: tuple = ('eu', 'ea', 'ua')  # e=emb, u=user, a=action
    sweep_learning_rates:   tuple = (0.005, 0.01, 0.03, 0.05)
    sweep_cb_types:         tuple = ('dr', 'mtr')
    sweep_data_repeats:     tuple = (1, 2)


def make_training_config(use_case_id: str | None = None) -> TrainingConfig:
    use_case_id = use_case_id or get_use_case_id('pl-aip-uplift')
    return TrainingConfig(
        input_path           = Path(f'/tmp/vw_training_{use_case_id}_FINAL.txt'),
        output_dir           = Path('task6_artifacts'),
        s3_bucket            = os.getenv('S3_CONFIG_BUCKET', 'aks-nvtabular-data'),
        s3_input_key         = f'training_data/vw_training_{use_case_id}_FINAL.txt',
        s3_output_prefix     = f'model_artifacts/{use_case_id}',
        propensity_audit_key = f'training_data/vw_propensity_audit_{use_case_id}.json',
        training_intent      = os.getenv('TRAINING_INTENT', 'cold_start_uniform'),
        alignment_key        = f'training_data/vw_training_alignment_{use_case_id}.parquet',
        vw_binary            = os.getenv('VW_BINARY') or None,
        cleanup_local        = True,
        random_seed          = 42,
        train_ratio          = 0.70,
        valid_ratio          = 0.15,
        min_actions_per_slate= 50,
        l2_regularization    = 1e-6,
        clip_p               = 0.02,
        quadratic_interactions = ('eu', 'ea', 'ua'),
        sweep_learning_rates = (0.005, 0.01, 0.03, 0.05),
        sweep_cb_types       = ('dr', 'mtr'),
        sweep_data_repeats   = (1, 2),
    )


cfg = make_training_config(USE_CASE_ID)
print(f'USE_CASE_ID             : {USE_CASE_ID}')
print(f'training_intent         : {cfg.training_intent}')
print(f'quadratic_interactions  : {cfg.quadratic_interactions}')
print(f'clip_p                  : {cfg.clip_p}')
print(f'l2_regularization       : {cfg.l2_regularization}')
print(f'sweep: {len(cfg.sweep_cb_types)} cb_types x {len(cfg.sweep_learning_rates)} LRs x {len(cfg.sweep_data_repeats)} repeats = {len(cfg.sweep_cb_types)*len(cfg.sweep_learning_rates)*len(cfg.sweep_data_repeats)} candidates')


In [ ]:
# Bootstrap imports so this cell runs standalone
import itertools, json, os, random, re, shutil
try:
    import boto3
except Exception:
    boto3 = None
import numpy as np, pandas as pd
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from scipy.stats import entropy as scipy_entropy
from typing import List

try:
    vowpalwabbit
except NameError:
    try:
        import vowpalwabbit
    except Exception:
        vowpalwabbit = None


def get_use_case_id(default: str = 'pl-aip-uplift') -> str:
    value = os.getenv('USE_CASE_ID', default).strip()
    if not value:
        raise ValueError('USE_CASE_ID must be non-empty.')
    return value


if 'USE_CASE_ID' not in globals():
    USE_CASE_ID = get_use_case_id('pl-aip-uplift')

if 'LABEL_RE' not in globals():
    # BUG 3 FIX: accept both '0:cost:prob |action' and 'cost:prob |action'
    LABEL_RE = re.compile(
        r'^\s*(?:0:)?(?P<cost>[0-9]*\.?[0-9]+):(?P<prob>[0-9]*\.?[0-9]+)\s+\|action\b'
    )

if 'TrainingConfig' not in globals():
    @dataclass
    class TrainingConfig:
        input_path: Path
        output_dir: Path
        s3_bucket: str
        s3_input_key: str
        s3_output_prefix: str
        propensity_audit_key: str
        training_intent: str
        alignment_key: str | None = None
        vw_binary: str | None = None
        cleanup_local: bool = True
        random_seed: int = 42
        train_ratio: float = 0.70
        valid_ratio: float = 0.15
        min_actions_per_slate: int = 50
        l2_regularization: float = 1e-6
        clip_p: float = 0.02
        quadratic_interactions: tuple = ('eu', 'ea', 'ua')
        sweep_learning_rates: tuple = (0.005, 0.01, 0.03, 0.05)
        sweep_cb_types: tuple = ('dr', 'mtr')
        sweep_data_repeats: tuple = (1, 2)


def make_training_config(use_case_id: str | None = None) -> TrainingConfig:
    use_case_id = use_case_id or get_use_case_id('pl-aip-uplift')
    return TrainingConfig(
        input_path=Path(f'/tmp/vw_training_{use_case_id}_FINAL.txt'),
        output_dir=Path('task6_artifacts'),
        s3_bucket=os.getenv('S3_CONFIG_BUCKET', 'aks-nvtabular-data'),
        s3_input_key=f'training_data/vw_training_{use_case_id}_FINAL.txt',
        s3_output_prefix=f'model_artifacts/{use_case_id}',
        propensity_audit_key=f'training_data/vw_propensity_audit_{use_case_id}.json',
        training_intent=os.getenv('TRAINING_INTENT', 'cold_start_uniform'),
        alignment_key=f'training_data/vw_training_alignment_{use_case_id}.parquet',
        vw_binary=os.getenv('VW_BINARY') or None,
        cleanup_local=True,
        random_seed=42,
        train_ratio=0.70,
        valid_ratio=0.15,
        min_actions_per_slate=50,
        l2_regularization=1e-6,
        clip_p=0.02,
        quadratic_interactions=('eu', 'ea', 'ua'),
        sweep_learning_rates=(0.005, 0.01, 0.03, 0.05),
        sweep_cb_types=('dr', 'mtr'),
        sweep_data_repeats=(1, 2),
    )


if 'cfg' not in globals():
    cfg = make_training_config(USE_CASE_ID)

print('Bootstrap imports/config loaded.')


# ═══════════════════════════════════════════════════════════
# HELPER FUNCTIONS
# ═══════════════════════════════════════════════════════════

def read_blocks(path: Path) -> List[str]:
    text = path.read_text(encoding='utf-8')
    return [b.strip('\n') for b in text.split('\n\n') if b.strip()]


def write_blocks(path: Path, blocks: List[str]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text('\n\n'.join(blocks) + '\n', encoding='utf-8')


def ensure_prefix(prefix: str) -> str:
    return prefix.rstrip('/')


def derive_run_id(use_case_id: str) -> str:
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    return f'{stamp}_{use_case_id.replace("-", "_")}'


def download_s3_to_local(s3_client, bucket: str, key: str, local_path: Path) -> None:
    local_path.parent.mkdir(parents=True, exist_ok=True)
    s3_client.download_file(bucket, key, str(local_path))


def download_optional_s3_to_local(s3_client, bucket: str, key: str | None, local_path: Path) -> bool:
    if not key:
        return False
    try:
        download_s3_to_local(s3_client, bucket, key, local_path)
        return True
    except Exception:
        return False


def upload_file_to_s3(s3_client, local_path: Path, bucket: str, key: str) -> None:
    s3_client.upload_file(str(local_path), bucket, key)


def load_alignment_frame(path: Path | None) -> pd.DataFrame | None:
    if path is None or not path.exists():
        return None
    df = pd.read_parquet(path)
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'])
    return df.reset_index(drop=True)


def normalize_training_mode(mode: str) -> str:
    mode = mode.strip().lower()
    allowed = {'strict_ope', 'cold_start_uniform'}
    if mode not in allowed:
        raise ValueError(f'training_intent must be one of {sorted(allowed)}, got {mode!r}.')
    return mode


def normalize_vw_blocks(path: Path) -> Path:
    lines = path.read_text(encoding='utf-8').splitlines()
    blocks, current = [], []
    for raw in lines:
        line = raw.strip()
        if not line:
            if current:
                blocks.append(current)
                current = []
            continue
        if line.startswith('shared '):
            if current:
                blocks.append(current)
            current = [line]
            continue
        if current:
            current.append(line)
    if current:
        blocks.append(current)

    strict_blocks, rejected = [], 0
    for b in blocks:
        if not b or not b[0].startswith('shared '):
            rejected += 1
            continue
        strict_blocks.append('\n'.join(b))

    if not strict_blocks:
        raise ValueError('normalize_vw_blocks: all blocks rejected. Re-run Task 5.')
    cleaned = path.with_name(path.stem + '_cleaned.txt')
    cleaned.write_text('\n\n'.join(strict_blocks) + '\n', encoding='utf-8')
    if rejected:
        print(f'  normalize_vw_blocks: kept {len(strict_blocks):,}, rejected {rejected:,}')
    return cleaned


def validate_block(block: str, block_index: int, min_actions: int) -> dict:
    lines = [l.rstrip() for l in block.splitlines() if l.strip()]
    if not lines:
        raise ValueError(f'Block {block_index} is empty.')
    if not lines[0].startswith('shared '):
        raise ValueError(f'Block {block_index} missing shared line.')
    action_lines = lines[1:]
    if not action_lines:
        raise ValueError(f'Block {block_index} has no action lines.')

    labeled = [i for i, l in enumerate(action_lines) if LABEL_RE.match(l)]
    if len(labeled) != 1:
        raise ValueError(f'Block {block_index}: expected 1 labeled action, got {len(labeled)}.')
    if labeled[0] != 0:
        raise ValueError(f'Block {block_index}: labeled action must be at index 0 (found {labeled[0]}).')
    if len(action_lines) < min_actions:
        raise ValueError(f'Block {block_index}: {len(action_lines)} actions < minimum {min_actions}.')

    m = LABEL_RE.match(action_lines[0])
    cost, prob = float(m.group('cost')), float(m.group('prob'))
    if not (0.0 <= cost <= 1.0):
        raise ValueError(f'Block {block_index}: cost {cost:.6f} not in [0,1].')
    if not (0.0 < prob <= 1.0):
        raise ValueError(f'Block {block_index}: prob {prob:.6f} not in (0,1].')
    return {'cost': cost, 'prob': prob, 'num_actions': len(action_lines)}


def validate_vw_file(path: Path, min_actions: int, sample_size: int | None = None):
    if not path.exists():
        raise FileNotFoundError(f'Input file not found: {path}')
    blocks = read_blocks(path)
    if not blocks:
        raise ValueError('No training examples in input file.')
    k = len(blocks) if sample_size is None else min(sample_size, len(blocks))
    indices = np.linspace(0, len(blocks) - 1, k, dtype=int)
    costs, probs, actions = [], [], []
    for i in indices:
        s = validate_block(blocks[i], int(i) + 1, min_actions)
        costs.append(s['cost']); probs.append(s['prob']); actions.append(s['num_actions'])
    return blocks, {
        'total_blocks': len(blocks), 'validated_blocks': len(indices),
        'cost_min': float(np.min(costs)), 'cost_max': float(np.max(costs)),
        'prob_min': float(np.min(probs)), 'prob_max': float(np.max(probs)),
        'action_min': int(np.min(actions)), 'action_max': int(np.max(actions)),
        'action_mean': float(np.mean(actions)),
    }


def split_blocks_for_training(blocks, train_ratio, valid_ratio, alignment_df=None):
    if alignment_df is None:
        raise ValueError('Production split requires alignment parquet with timestamps.')
    if len(alignment_df) != len(blocks):
        raise ValueError(f'Alignment rows ({len(alignment_df):,}) != VW blocks ({len(blocks):,}). Re-run Task 5.')
    if 'timestamp' not in alignment_df.columns:
        raise ValueError('Alignment artifact missing timestamp column.')

    ordered = alignment_df.copy()
    ordered['timestamp'] = pd.to_datetime(ordered['timestamp'], errors='coerce')
    invalid_ts = int(ordered['timestamp'].isna().sum())
    if invalid_ts:
        raise ValueError(f'{invalid_ts:,} invalid timestamps in alignment artifact.')

    ordered['block_text'] = blocks
    sort_cols = [c for c in ['timestamp', 'campaign_id', 'master_user_id', 'event_order'] if c in ordered.columns]
    ordered = ordered.sort_values(sort_cols, kind='stable').reset_index(drop=True)

    n = len(ordered)
    train_end = int(n * train_ratio)
    valid_end = int(n * (train_ratio + valid_ratio))
    if train_end <= 0 or valid_end <= train_end or valid_end >= n:
        raise ValueError('Invalid split sizes — all three splits must be non-empty.')

    train_df = ordered.iloc[:train_end].copy()
    valid_df = ordered.iloc[train_end:valid_end].copy()
    test_df  = ordered.iloc[valid_end:].copy()
    strict   = bool(train_df['timestamp'].max() < valid_df['timestamp'].min()
                    and valid_df['timestamp'].max() < test_df['timestamp'].min())

    return (
        train_df['block_text'].tolist(),
        valid_df['block_text'].tolist(),
        test_df['block_text'].tolist(),
        {
            'strategy':                   'chronological_train_validation_test',
            'alignment_rows':             n,
            'strict_temporal_boundaries': strict,
            'train_ratio':                train_ratio,
            'validation_ratio':           valid_ratio,
            'test_ratio':                 1.0 - train_ratio - valid_ratio,
            'train_rows':                 len(train_df),
            'validation_rows':            len(valid_df),
            'test_rows':                  len(test_df),
            'train_start':                train_df['timestamp'].iloc[0].isoformat(),
            'train_end':                  train_df['timestamp'].iloc[-1].isoformat(),
            'validation_start':           valid_df['timestamp'].iloc[0].isoformat(),
            'validation_end':             valid_df['timestamp'].iloc[-1].isoformat(),
            'test_start':                 test_df['timestamp'].iloc[0].isoformat(),
            'test_end':                   test_df['timestamp'].iloc[-1].isoformat(),
        },
    )


def candidate_id(spec: dict) -> str:
    lr = str(spec['learning_rate']).replace('.', 'p')
    return f"cb_{spec['cb_type']}_lr_{lr}_rep_{spec['data_repeat']}"


def build_candidate_grid(cfg: TrainingConfig) -> list:
    return [
        {'cb_type': ct, 'learning_rate': lr, 'data_repeat': dr}
        for ct, lr, dr in itertools.product(cfg.sweep_cb_types, cfg.sweep_learning_rates, cfg.sweep_data_repeats)
    ]


def is_cold_start_replay(alignment_df) -> bool:
    if alignment_df is None or 'decision_policy' not in alignment_df.columns:
        return False
    policies = set(alignment_df['decision_policy'].dropna().astype(str).unique())
    return bool(policies) and policies <= {'cold_start_uniform'}


def load_propensity_audit(s3_client, bucket, key):
    try:
        obj = s3_client.get_object(Bucket=bucket, Key=key)
        return json.loads(obj['Body'].read().decode('utf-8'))
    except Exception:
        return None


def propensity_mode_summary(audit):
    if not audit:
        return {'audit_found': False, 'strict_ope_ready': False, 'source_counts': {}}
    source_counts = audit.get('source_counts', {})
    strict_ope_ready = bool(source_counts) and all(str(k).startswith('column:') for k in source_counts.keys())
    return {
        'audit_found':                True,
        'strict_ope_ready':           bool(strict_ope_ready),
        'source_counts':              source_counts,
        'has_true_logged_propensity': bool(audit.get('has_true_logged_propensity', False)),
        'has_non_logged_propensity':  bool(audit.get('has_non_logged_propensity', False)),
    }


def analyze_concentration(values, universe_size=None):
    clean = [int(v) for v in values if v is not None and pd.notna(v)]
    if not clean:
        return {'unique_items': 0, 'normalized_entropy': 0.0, 'herfindahl': 1.0,
                'top1_concentration': 1.0, 'top5_concentration': 1.0, 'coverage': 0.0}
    counts = pd.Series(clean).value_counts()
    probs  = (counts / counts.sum()).values.astype(float)
    max_h  = np.log(max(len(counts), 1))
    denom  = max(int(universe_size or len(counts)), 1)
    return {
        'unique_items':        int(len(counts)),
        'normalized_entropy':  float(scipy_entropy(probs) / max_h) if max_h > 0 else 0.0,
        'herfindahl':          float(np.sum(probs ** 2)),
        'top1_concentration':  float(probs[:1].sum()),
        'top5_concentration':  float(probs[:5].sum()),
        'coverage':            float(min(len(counts) / denom, 1.0)),
    }


def apply_candidate_gates(result: dict) -> tuple:
    failures = []
    if result.get('status') != 'completed':        failures.append('candidate_not_completed')
    for k in ['train_average_loss', 'valid_ips_reward']:
        if result.get(k) is None or not np.isfinite(float(result.get(k, 0))):
            failures.append(f'{k}_not_finite')
    if float(result.get('dr_vs_historical_pct', -1e9)) < 0.0:
        failures.append('dr_lift_negative')
    pos = result.get('position_concentration') or {}
    aid = result.get('action_id_concentration') or {}
    if float(pos.get('normalized_entropy', 0.0)) < 0.30:  failures.append('position_entropy_below_floor')
    if float(aid.get('normalized_entropy', 0.0)) < 0.30:  failures.append('action_id_entropy_below_floor')
    if float(aid.get('top1_concentration', 1.0)) > 0.35:  failures.append('action_id_top1_above_cap')
    if float(aid.get('top5_concentration', 1.0)) > 0.80:  failures.append('action_id_top5_above_cap')
    if float(pos.get('coverage', 0.0)) <= 0.20:           failures.append('position_coverage_below_floor')
    return not failures, failures


print('Helper functions loaded.')


In [ ]:
# ===================================================================
# VW TRAINING + EVALUATION (Python package if available, CLI fallback)
# ===================================================================

import os
import re
import shutil
import subprocess
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd

try:
    vowpalwabbit
except NameError:
    try:
        import vowpalwabbit
    except Exception:
        vowpalwabbit = None


def find_vw_binary(vw_binary: str | None = None) -> str:
    if vw_binary:
        candidate = Path(vw_binary)
        if candidate.exists():
            return str(candidate)
    vw_path = shutil.which('vw')
    if vw_path:
        return vw_path
    for path in ['/usr/bin/vw', '/bin/vw', '/opt/conda/bin/vw']:
        if Path(path).exists():
            return path
    raise RuntimeError(
        'VW is not available. Install either the Python package `vowpalwabbit` '
        'or the CLI package that provides the `vw` command.'
    )


def vw_backend(cfg=None) -> str:
    if vowpalwabbit is not None:
        return 'python'
    find_vw_binary(getattr(cfg, 'vw_binary', None))
    return 'cli'


def vw_cli_supports_option(vw_bin: str, option: str) -> bool:
    try:
        proc = subprocess.run([vw_bin, '--help'], capture_output=True, text=True, timeout=20)
        help_text = (proc.stdout or '') + '\n' + (proc.stderr or '')
        return option in help_text
    except Exception:
        return False


def maybe_add_cli_option(args: list[str], vw_bin: str, option: str, value: str | None = None) -> None:
    if vw_cli_supports_option(vw_bin, option):
        args.append(option)
        if value is not None:
            args.append(value)
    else:
        suffix = f' {value}' if value is not None else ''
        print(f'  VW CLI does not support {option}{suffix}; omitting it for model compatibility.')


def build_vw_arg_list(spec: dict, cfg: TrainingConfig, model_path: Path | None = None, vw_bin: str | None = None) -> list[str]:
    args = [
        '--cb_explore_adf',
        '--epsilon', '0.0',
        '--cb_type', str(spec['cb_type']),
        '--holdout_off',
        '--random_seed', str(cfg.random_seed),
        '--l2', str(cfg.l2_regularization),
        '-l', str(spec['learning_rate']),
    ]

    if vw_bin is None:
        # Python package path. Keep the full modern argument set.
        args.extend(['--clip_p', str(cfg.clip_p)])
        args.extend(['--cb_min_cost', '0.0', '--cb_max_cost', '1.0'])
    else:
        # CLI path. Ubuntu's vw 8.6.1 does not support --clip_p, and a model saved
        # with unsupported options may not reload in Task 7. Only pass options the CLI knows.
        maybe_add_cli_option(args, vw_bin, '--clip_p', str(cfg.clip_p))
        maybe_add_cli_option(args, vw_bin, '--cb_min_cost', '0.0')
        maybe_add_cli_option(args, vw_bin, '--cb_max_cost', '1.0')

    args.extend(['--power_t', '0'])
    if model_path is not None:
        args.extend(['-f', str(model_path)])
    for interaction in cfg.quadratic_interactions:
        args.extend(['-q', str(interaction)])
    return args


def build_vw_args(spec: dict, cfg: TrainingConfig, model_path: Path) -> str:
    return ' '.join(build_vw_arg_list(spec, cfg, model_path, vw_bin=None))


def run_vw_cli(cmd: list[str], log_path: Path | None = None) -> str:
    proc = subprocess.run(cmd, capture_output=True, text=True)
    output = (proc.stdout or '') + (('\n' + proc.stderr) if proc.stderr else '')
    if log_path is not None:
        log_path.parent.mkdir(parents=True, exist_ok=True)
        log_path.write_text(output, encoding='utf-8')
    if proc.returncode != 0:
        raise RuntimeError(f'VW command failed with exit code {proc.returncode}.\n{output[-4000:]}')
    return output


def canonicalize_vw_label_line(line: str) -> str:
    stripped = line.strip()
    if re.match(r'^\s*0:[0-9]*\.?[0-9]+:[0-9]*\.?[0-9]+\s+\|action\b', stripped):
        return stripped
    m = re.match(
        r'^\s*(?P<cost>[0-9]*\.?[0-9]+):(?P<prob>[0-9]*\.?[0-9]+)(?P<rest>\s+\|action\b.*)$',
        stripped,
    )
    if m:
        return f"0:{m.group('cost')}:{m.group('prob')}{m.group('rest')}"
    return stripped


def canonicalize_vw_block(block: str) -> str:
    return '\n'.join(canonicalize_vw_label_line(line) for line in block.splitlines() if line.strip())


def ensure_documented_adf_blocks(blocks: list[str]) -> list[str]:
    return [canonicalize_vw_block(block) for block in blocks]


def mean_logged_cost(blocks: list[str]) -> float:
    costs = []
    for block in blocks:
        for line in block.splitlines():
            m = LABEL_RE.match(line.strip())
            if m:
                costs.append(float(m.group('cost')))
                break
    return float(np.mean(costs)) if costs else 0.5


def parse_block_for_eval(block: str) -> dict:
    lines = [l.strip() for l in block.splitlines() if l.strip()]
    act_lines = lines[1:] if len(lines) > 1 else []
    action_aids = []
    for line in act_lines:
        m = re.search(r'\baid_(\d+)\b', line)
        action_aids.append(int(m.group(1)) if m else None)

    parsed = {
        'cost': None,
        'log_prob': None,
        'chosen_idx': None,
        'chosen_aid': None,
        'reward': None,
        'num_actions': len(act_lines),
        'action_aids': action_aids,
    }
    for i, line in enumerate(act_lines):
        m = LABEL_RE.match(line)
        if m:
            parsed['cost'] = float(m.group('cost'))
            parsed['log_prob'] = float(m.group('prob'))
            parsed['chosen_idx'] = i
            parsed['reward'] = 1.0 - float(m.group('cost'))
            if 0 <= i < len(action_aids):
                parsed['chosen_aid'] = action_aids[i]
            break
    return parsed


def strip_labels_from_block(block: str) -> str:
    out = []
    for line in block.splitlines():
        s = line.strip()
        if LABEL_RE.match(s):
            s = re.sub(r'^\s*(?:0:)?[0-9.]+:[0-9.]+\s+', '', s)
        if s:
            out.append(s)
    return '\n'.join(out)


def write_unlabeled_blocks(path: Path, blocks: list[str]) -> None:
    stripped = [strip_labels_from_block(block) for block in blocks]
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text('\n\n'.join(stripped) + '\n', encoding='utf-8')


def normalize_raw_action_index(raw: int, all_raw: list[int] | None, num_actions: int) -> int | None:
    if all_raw:
        mn, mx = min(all_raw), max(all_raw)
        if mn >= 1 and mx <= num_actions:
            return raw - 1
        if mn >= 0 and mx < num_actions:
            return raw
    if raw == 0:
        return 0
    if 1 <= raw <= num_actions:
        return raw - 1
    if 0 <= raw < num_actions:
        return raw
    return None


def prediction_to_position(prediction, num_actions: int) -> int:
    if prediction is None:
        return 0
    if isinstance(prediction, (int, float, np.integer, np.floating)):
        pos = normalize_raw_action_index(int(prediction), None, num_actions)
        return 0 if pos is None else pos
    if isinstance(prediction, list) and prediction:
        first = prediction[0]
        if hasattr(first, 'action') and hasattr(first, 'score'):
            raw_actions = [int(item.action) for item in prediction]
            best = max(prediction, key=lambda item: float(item.score))
            pos = normalize_raw_action_index(int(best.action), raw_actions, num_actions)
            return 0 if pos is None else pos
        if isinstance(first, (tuple, list)) and len(first) >= 2:
            raw_actions = [int(item[0]) for item in prediction]
            best = max(prediction, key=lambda item: float(item[1]))
            pos = normalize_raw_action_index(int(best[0]), raw_actions, num_actions)
            return 0 if pos is None else pos
        return int(np.argmax([float(x) for x in prediction]))
    return 0


def parse_prediction_line(line: str, num_actions: int) -> int | None:
    s = line.strip()
    if not s:
        return None
    pair_text = s.replace(',', ' ')
    pairs = re.findall(r'(?<!\S)(\d+):([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)(?=\s|$)', pair_text)
    if pairs:
        raw_actions = [int(action) for action, _ in pairs]
        best_raw = int(max(pairs, key=lambda pair: float(pair[1]))[0])
        return normalize_raw_action_index(best_raw, raw_actions, num_actions)
    tokens = s.split()
    numbers = []
    for token in tokens:
        try:
            numbers.append(float(token.split(',', 1)[0]))
        except ValueError:
            pass
    if len(numbers) >= num_actions:
        return int(np.argmax(numbers[:num_actions]))
    if numbers:
        raw_float = numbers[0]
        raw = int(raw_float)
        if raw_float == raw:
            return normalize_raw_action_index(raw, None, num_actions)
    return None


def read_cli_predictions(pred_path: Path, expected_counts: list[int]) -> list[int]:
    raw_lines = pred_path.read_text(encoding='utf-8').splitlines()
    nonblank = [line.strip() for line in raw_lines if line.strip()]
    parsed = []
    for line in nonblank:
        if len(parsed) >= len(expected_counts):
            break
        pos = parse_prediction_line(line, expected_counts[len(parsed)])
        if pos is not None:
            parsed.append(pos)
    if len(parsed) == len(expected_counts):
        return parsed
    raise RuntimeError(
        f'Could not parse VW predictions: got {len(parsed)}, expected {len(expected_counts)}. '
        f'First prediction lines: {nonblank[:5]}'
    )


def train_candidate(blocks: list[str], model_path: Path, spec: dict, cfg: TrainingConfig) -> dict:
    blocks = ensure_documented_adf_blocks(blocks)
    backend = vw_backend(cfg)
    model_path.parent.mkdir(parents=True, exist_ok=True)
    costs = []

    if backend == 'python':
        args_str = build_vw_args(spec, cfg, model_path)
        print('  VW backend: python package')
        print(f'  VW args: {args_str}')
        vw = vowpalwabbit.Workspace(args_str, quiet=True)
        for block in blocks:
            vw.learn(block)
            for line in block.splitlines():
                m = LABEL_RE.match(line.strip())
                if m:
                    costs.append(float(m.group('cost')))
                    break
        n = vw.get_weighted_examples()
        total_loss = vw.get_sum_loss()
        vw.finish()
    else:
        vw_bin = find_vw_binary(getattr(cfg, 'vw_binary', None))
        with tempfile.TemporaryDirectory() as tmp:
            train_path = Path(tmp) / 'train.vw'
            write_blocks(train_path, blocks)
            cmd = [vw_bin, '-d', str(train_path), *build_vw_arg_list(spec, cfg, model_path, vw_bin=vw_bin), '--quiet']
            print(f'  VW backend: cli ({vw_bin})')
            print('  VW args: ' + ' '.join(cmd[1:]))
            run_vw_cli(cmd, model_path.parent / 'vw_train.log')
        n = len(blocks)
        total_loss = 0.0
        costs = [mean_logged_cost(blocks)]

    avg_loss = float(np.mean(costs)) if costs else mean_logged_cost(blocks)
    return {
        'weighted_examples': float(n),
        'sum_loss': float(total_loss),
        'train_average_loss': avg_loss,
        'vw_backend': backend,
    }


def predict_positions(model_path: Path, valid_blocks: list[str], spec: dict, cfg: TrainingConfig) -> list[int]:
    valid_blocks = ensure_documented_adf_blocks(valid_blocks)
    parsed_counts = [max(parse_block_for_eval(block)['num_actions'], 1) for block in valid_blocks]
    backend = vw_backend(cfg)

    if backend == 'python':
        eval_args = f'--cb_explore_adf -t -i {str(model_path)}'
        vw_eval = vowpalwabbit.Workspace(eval_args, quiet=True)
        predicted = []
        for block, num_actions in zip(valid_blocks, parsed_counts):
            stripped_block = strip_labels_from_block(block)
            if stripped_block.strip():
                pred_result = vw_eval.predict(stripped_block)
                predicted.append(prediction_to_position(pred_result, num_actions))
            else:
                predicted.append(0)
        vw_eval.finish()
        return predicted

    vw_bin = find_vw_binary(getattr(cfg, 'vw_binary', None))
    with tempfile.TemporaryDirectory() as tmp:
        pred_input = Path(tmp) / 'valid_unlabeled.vw'
        pred_output = Path(tmp) / 'predictions.txt'
        write_unlabeled_blocks(pred_input, valid_blocks)
        cmd = [
            vw_bin, '--cb_explore_adf', '-t', '-i', str(model_path), '-d', str(pred_input),
            '-p', str(pred_output), '--quiet',
        ]
        run_vw_cli(cmd, model_path.parent / 'vw_valid_predict.log')
        return read_cli_predictions(pred_output, parsed_counts)


def evaluate_candidate(model_path: Path, valid_blocks: list[str], train_blocks: list[str], spec: dict, cfg: TrainingConfig, slate_size: int) -> dict:
    valid_blocks = ensure_documented_adf_blocks(valid_blocks)
    train_blocks = ensure_documented_adf_blocks(train_blocks)
    parsed = [parse_block_for_eval(b) for b in valid_blocks]
    parse_failures = sum(1 for p in parsed if p['reward'] is None or p['log_prob'] is None)
    if parse_failures:
        raise RuntimeError(f'Could not parse {parse_failures:,} validation blocks. Check LABEL_RE.')

    predicted_positions = predict_positions(model_path, valid_blocks, spec, cfg)
    if len(predicted_positions) != len(parsed):
        raise RuntimeError(f'Pred count ({len(predicted_positions)}) != blocks ({len(parsed)}).')

    rewards = np.array([p['reward'] for p in parsed], dtype=float)
    log_prob = np.array([p['log_prob'] for p in parsed], dtype=float)
    logged_aid = [p['chosen_aid'] for p in parsed]
    pred_aid = [
        p['action_aids'][pos] if p['action_aids'] and 0 <= pos < len(p['action_aids']) else None
        for pos, p in zip(predicted_positions, parsed)
    ]
    match = np.array([
        1.0 if (la is not None and pa is not None and la == pa) else 0.0
        for la, pa in zip(logged_aid, pred_aid)
    ], dtype=float)
    hist_val = float(rewards.mean())

    valid_lp = np.isfinite(log_prob) & (log_prob > 0.0) & (log_prob <= 1.0)
    clipped_lp = np.where(valid_lp, np.maximum(log_prob, cfg.clip_p), np.nan)
    ips_w = np.where(valid_lp, match / clipped_lp, 0.0)
    ips_reward = float((ips_w * rewards).mean())

    train_parsed = [parse_block_for_eval(b) for b in train_blocks]
    usable = [p for p in train_parsed if p['reward'] is not None and p['chosen_aid'] is not None]
    if usable:
        dm_df = pd.DataFrame({'aid': [p['chosen_aid'] for p in usable], 'reward': [p['reward'] for p in usable]})
        arm_reward = dm_df.groupby('aid')['reward'].mean().to_dict()
        global_rew = float(dm_df['reward'].mean())
    else:
        arm_reward, global_rew = {}, float(rewards.mean())

    r_hat_pred = pd.Series(pred_aid).map(arm_reward).fillna(global_rew).values
    r_hat_log = pd.Series(logged_aid).map(arm_reward).fillna(global_rew).values
    dr_value = float((r_hat_pred + ips_w * (rewards - r_hat_log)).mean())
    dr_vs_hist = ((dr_value - hist_val) / max(abs(hist_val), 1e-6)) * 100.0

    action_universe = len({aid for p in parsed for aid in p['action_aids'] if aid is not None})
    pos_conc = analyze_concentration(predicted_positions, universe_size=slate_size)
    aid_conc = analyze_concentration(pred_aid, universe_size=action_universe)

    failures = []
    if dr_vs_hist <= 0.0: failures.append('dr_lift_not_positive')
    if pos_conc['normalized_entropy'] < 0.30: failures.append('position_entropy_below_floor')
    if pos_conc['top5_concentration'] > 0.80: failures.append('position_top5_above_cap')
    if pos_conc['coverage'] <= 0.20: failures.append('position_coverage_below_floor')
    if aid_conc['normalized_entropy'] < 0.30: failures.append('action_id_entropy_below_floor')
    if aid_conc['top1_concentration'] > 0.35: failures.append('action_id_top1_above_cap')
    if aid_conc['top5_concentration'] > 0.80: failures.append('action_id_top5_above_cap')

    return {
        'historical_value': hist_val,
        'ips_reward': ips_reward,
        'valid_ips_reward': ips_reward,
        'dr_value': dr_value,
        'dr_vs_historical_pct': float(dr_vs_hist),
        'match_rate': float(match.mean()),
        'position_concentration': pos_conc,
        'action_id_concentration': aid_conc,
        'promotion_passed': not failures,
        'promotion_failures': failures,
        'safety_score': float(
            max(pos_conc['normalized_entropy'], 0) + max(aid_conc['normalized_entropy'], 0)
            + max(pos_conc['coverage'], 0) + max(dr_vs_hist / 100.0, -1.0)
        ),
        'vw_backend': vw_backend(cfg),
    }

print('VW training/evaluation functions loaded.')
try:
    print(f'VW backend available: {vw_backend(cfg)}')
except Exception as e:
    print(f'VW backend not available yet: {e}')


In [ ]:
# ═══════════════════════════════════════════════════════════
# MAIN TRAINING LOOP  (unchanged from v1 except function calls above)
# ═══════════════════════════════════════════════════════════

required_setup = [
    'boto3', 'get_use_case_id', 'make_training_config', 'ensure_prefix',
    'normalize_training_mode', 'train_candidate', 'evaluate_candidate',
]
missing_setup = [name for name in required_setup if name not in globals()]
if missing_setup:
    raise RuntimeError(f'Run setup cells first. Missing: {missing_setup}')

if 'USE_CASE_ID' not in globals():
    USE_CASE_ID = get_use_case_id('pl-aip-uplift')
if 'cfg' not in globals():
    cfg = make_training_config(USE_CASE_ID)

s3_client        = boto3.client('s3')
base_prefix      = ensure_prefix(cfg.s3_output_prefix)
run_id           = derive_run_id(USE_CASE_ID)
prefix           = f'{base_prefix}/runs/{run_id}'
alignment_local  = cfg.output_dir / f'vw_training_alignment_{USE_CASE_ID}.parquet'

mode             = normalize_training_mode(cfg.training_intent)
propensity_audit = load_propensity_audit(s3_client, cfg.s3_bucket, cfg.propensity_audit_key)
prop_summary     = propensity_mode_summary(propensity_audit)

print(f'Training intent    : {mode}')
print(f'Propensity audit   : {prop_summary["audit_found"]}')
print(f'strict_ope_ready   : {prop_summary["strict_ope_ready"]}')
print(f'Quadratics         : {cfg.quadratic_interactions}')
print(f'clip_p             : {cfg.clip_p}')
print(f'L2                 : {cfg.l2_regularization}')

if mode == 'strict_ope' and not prop_summary['strict_ope_ready']:
    raise ValueError('strict_ope requires strict_ope_ready=True. Re-run Task 5 with real logged propensities.')
elif mode == 'cold_start_uniform' and not prop_summary['strict_ope_ready']:
    print('Cold-start mode: synthetic propensities allowed. production_promotable will remain False.')

download_s3_to_local(s3_client, cfg.s3_bucket, cfg.s3_input_key, cfg.input_path)
clean_input_path = normalize_vw_blocks(cfg.input_path)

alignment_found  = download_optional_s3_to_local(s3_client, cfg.s3_bucket, cfg.alignment_key, alignment_local)
if not alignment_found:
    raise ValueError('Alignment parquet required for chronological split. Re-run Task 5.')
alignment_df     = load_alignment_frame(alignment_local)
cold_start       = (mode == 'cold_start_uniform') or is_cold_start_replay(alignment_df)
if cold_start:
    print('Cold-start replay detected: selecting best validation candidate. production_promotable=False.')

blocks, val_summary = validate_vw_file(clean_input_path, min_actions=cfg.min_actions_per_slate, sample_size=None)
print(f'Validated {val_summary["validated_blocks"]:,}/{val_summary["total_blocks"]:,} blocks')
print(f'Cost [{val_summary["cost_min"]:.4f},{val_summary["cost_max"]:.4f}] | '
      f'Prob ({val_summary["prob_min"]:.6f},{val_summary["prob_max"]:.6f}] | '
      f'Actions min={val_summary["action_min"]}')

cfg.output_dir.mkdir(parents=True, exist_ok=True)
train_blocks, valid_blocks, test_blocks, split_summary = split_blocks_for_training(
    blocks, cfg.train_ratio, cfg.valid_ratio, alignment_df=alignment_df
)
print(f'Split: {len(train_blocks):,} train / {len(valid_blocks):,} valid / {len(test_blocks):,} test')
slate_size = int(val_summary['action_max'])

candidate_grid   = build_candidate_grid(cfg)
print(f'\nCandidate sweep: {len(candidate_grid)} configs')

candidate_results = []
for idx, spec in enumerate(candidate_grid, start=1):
    cand_id  = candidate_id(spec)
    cand_dir = cfg.output_dir / 'candidates' / cand_id
    cand_dir.mkdir(parents=True, exist_ok=True)

    rng            = random.Random(cfg.random_seed + idx)
    train_repeated = train_blocks * spec['data_repeat']
    rng.shuffle(train_repeated)

    model_path = cand_dir / f'vw_model_{USE_CASE_ID}.vw'

    result = {
        'candidate_id':                  cand_id,
        'status':                        'failed',
        'cb_type':                       spec['cb_type'],
        'learning_rate':                 spec['learning_rate'],
        'data_repeat':                   spec['data_repeat'],
        'num_train_examples_after_repeat': len(train_repeated),
    }

    try:
        print(f'\n[{idx}/{len(candidate_grid)}] Training {cand_id}')

        train_metrics = train_candidate(train_repeated, model_path, spec, cfg)
        train_loss    = train_metrics['train_average_loss']
        print(f'  Train: n={train_metrics["weighted_examples"]:.0f}  avg_loss={train_loss:.4f}')

        eval_metrics = evaluate_candidate(model_path, valid_blocks, train_blocks, spec, cfg, slate_size)
        valid_ips    = eval_metrics['valid_ips_reward']
        print(f'  Valid: IPS_reward={valid_ips:.4f}  DR_lift={eval_metrics["dr_vs_historical_pct"]:+.1f}%  '
              f'gates={"PASS" if eval_metrics["promotion_passed"] else "FAIL"}')

        result['train_average_loss'] = float(train_loss)
        result['valid_average_loss'] = float(valid_ips)
        result['valid_ips_reward']   = float(valid_ips)
        result['selection_score']    = float(-valid_ips)
        result.update(eval_metrics)
        result['status'] = 'completed'

        gate_passed, gate_failures = apply_candidate_gates(result)
        result['promotion_passed']   = gate_passed
        result['promotion_failures'] = gate_failures

        cand_metrics_path = cand_dir / 'candidate_metrics.json'
        cand_metrics_path.write_text(json.dumps(result, indent=2), encoding='utf-8')
        upload_file_to_s3(s3_client, model_path, cfg.s3_bucket, f'{prefix}/candidates/{cand_id}/vw_model_{USE_CASE_ID}.vw')
        upload_file_to_s3(s3_client, cand_metrics_path, cfg.s3_bucket, f'{prefix}/candidates/{cand_id}/candidate_metrics.json')

    except Exception as e:
        result['error'] = str(e)
        print(f'  FAILED: {e}')

    candidate_results.append(result)

successful  = [r for r in candidate_results if r['status'] == 'completed']
gate_pass   = [r for r in successful if r.get('promotion_passed')]

if not successful:
    raise RuntimeError('All candidates failed. Check VW args and training data.')

if gate_pass:
    best_candidate   = max(gate_pass, key=lambda r: float(r.get('valid_ips_reward', -1e9)))
    selection_mode   = 'promotion_gate_pass'
    task6_val_passed = True
elif cold_start:
    print('WARNING: No candidate passed all gates (expected for cold-start). Selecting by IPS reward.')
    best_candidate   = max(successful, key=lambda r: float(r.get('valid_ips_reward', -1e9)))
    selection_mode   = 'cold_start_replay_ips_best'
    task6_val_passed = False
else:
    raise RuntimeError('No candidate passed gates. Do not promote to production.')

print(f'\nSelected: {best_candidate["candidate_id"]}')
print(f'  train_loss={best_candidate["train_average_loss"]:.4f}')
print(f'  valid_ips_reward={best_candidate["valid_ips_reward"]:.4f}')
print(f'  DR_lift={best_candidate.get("dr_vs_historical_pct", 0):+.1f}%')
print(f'  selection_mode={selection_mode}')


In [ ]:
# ── Copy selected model + save all artifacts ─────────────────────────────────
selected_model_path = cfg.output_dir / f'vw_model_{USE_CASE_ID}.vw'
selected_cand_dir   = cfg.output_dir / 'candidates' / best_candidate['candidate_id']
shutil.copy2(selected_cand_dir / f'vw_model_{USE_CASE_ID}.vw', selected_model_path)

split_manifest_path    = cfg.output_dir / 'split_manifest.json'
candidate_sweep_path   = cfg.output_dir / 'candidate_sweep.json'
metrics_path           = cfg.output_dir / 'vw_metrics.json'
model_card_path        = cfg.output_dir / 'model_card.json'

split_manifest_path.write_text(json.dumps(split_summary, indent=2), encoding='utf-8')
candidate_sweep_path.write_text(json.dumps(candidate_results, indent=2), encoding='utf-8')

latest_manifest = {
    'run_id':                   run_id,
    'use_case_id':              USE_CASE_ID,
    's3_run_prefix':            f's3://{cfg.s3_bucket}/{prefix}',
    'created_at_utc':           datetime.now(timezone.utc).isoformat(),
    'training_intent':          mode,
    'selected_candidate_id':    best_candidate['candidate_id'],
    'selection_mode':           selection_mode,
    'promotion_passed':         bool(best_candidate.get('promotion_passed', False)),
    'production_promotable':    False,
    'task6_validation_passed':  bool(task6_val_passed),
    'cold_start_replay':        bool(cold_start),
}

metrics = {
    **latest_manifest,
    'cb_type':                  best_candidate['cb_type'],
    'learning_rate':            best_candidate['learning_rate'],
    'l2_regularization':        cfg.l2_regularization,
    'clip_p':                   cfg.clip_p,
    'quadratic_interactions':   list(cfg.quadratic_interactions),
    'random_seed':              cfg.random_seed,
    'data_repeat':              best_candidate['data_repeat'],
    'train_average_loss':       best_candidate['train_average_loss'],
    'valid_ips_reward':         best_candidate['valid_ips_reward'],
    'dr_vs_historical_pct':     best_candidate.get('dr_vs_historical_pct'),
    'match_rate':               best_candidate.get('match_rate'),
    'promotion_failures':       best_candidate.get('promotion_failures', []),
    'safety_score':             best_candidate.get('safety_score'),
    'position_concentration':   best_candidate.get('position_concentration'),
    'action_id_concentration':  best_candidate.get('action_id_concentration'),
    'num_blocks_total':         len(blocks),
    'num_blocks_train':         len(train_blocks),
    'num_blocks_valid':         len(valid_blocks),
    'num_blocks_test':          len(test_blocks),
    'split_summary':            split_summary,
    'validation_summary':       val_summary,
    'candidate_results':        candidate_results,
    'propensity_summary':       prop_summary,
}

model_card = {
    'use_case_id':           USE_CASE_ID,
    'run_id':                run_id,
    'created_at_utc':        datetime.now(timezone.utc).isoformat(),
    'training_mode':         mode,
    'strict_ope_ready':      bool(prop_summary.get('strict_ope_ready', False)),
    'production_promotable': False,
    'known_limitations': [] if task6_val_passed else [
        'Cold-start: synthetic propensities used. Not production-promotable from Task 6 alone.',
        'Run Task 7 on test split to confirm DR lift before promoting.'
    ],
}

metrics_path.write_text(json.dumps(metrics, indent=2), encoding='utf-8')
model_card_path.write_text(json.dumps(model_card, indent=2), encoding='utf-8')

upload_file_to_s3(s3_client, selected_model_path, cfg.s3_bucket, f'{prefix}/vw_model_{USE_CASE_ID}.vw')
upload_file_to_s3(s3_client, metrics_path,        cfg.s3_bucket, f'{prefix}/vw_metrics.json')
upload_file_to_s3(s3_client, split_manifest_path, cfg.s3_bucket, f'{prefix}/split_manifest.json')
upload_file_to_s3(s3_client, candidate_sweep_path,cfg.s3_bucket, f'{prefix}/candidate_sweep.json')
upload_file_to_s3(s3_client, model_card_path,     cfg.s3_bucket, f'{prefix}/model_card.json')

s3_client.put_object(
    Bucket=cfg.s3_bucket,
    Key=f'{base_prefix}/latest_run.json',
    Body=json.dumps(latest_manifest, indent=2).encode('utf-8'),
)

if cfg.cleanup_local:
    shutil.rmtree(cfg.output_dir, ignore_errors=True)
    for p in [cfg.input_path, clean_input_path, alignment_local]:
        try:
            if p and p.exists():
                p.unlink()
        except Exception:
            pass
    print('Local artifacts cleaned up.')

print('\n' + '=' * 70)
print('TASK 6 COMPLETE')
print('=' * 70)
print(f'  Run ID            : {run_id}')
print(f'  Selected candidate: {best_candidate["candidate_id"]}')
print(f'  Train loss        : {best_candidate["train_average_loss"]:.4f}')
print(f'  Valid IPS reward  : {best_candidate["valid_ips_reward"]:.4f}')
print(f'  DR lift           : {best_candidate.get("dr_vs_historical_pct", 0):+.1f}%')
print(f'  task6_val_passed  : {task6_val_passed}')
print(f'  production_promotable: False (Task 7 test-split eval required)')


In [ ]:
print('ALL BUGS FIXED IN THIS VERSION (v2):')
print()
print('BUG 1 (CRITICAL — RuntimeError crash): vw.learn(lines) — list[str] input')
print('  OLD: split block into lines list → vw.learn(lines)  ← CRASHES')
print('  WHY: cb_adf Python API requires full multiline block string, not list[str]')
print('  FIX: vw.learn(block)  — pass the whole block string with embedded newlines')
print()
print('BUG 2 (CRITICAL — wrong predictions): strip_labels returns list, predict gets list')
print('  OLD: strip_labels_from_block() returned list[str]; vw.predict(list) → wrong')
print('  OLD: np.argmax on tuples [(action_idx, prob)] was incorrect')
print('  FIX: strip_labels returns joined string; predict result parsed as (idx, prob) tuples')
print()
print('BUG 3 (CRITICAL — parse failures): LABEL_RE did not match Task 5 output')
print('  OLD: r"^\\s*cost:prob \\|action"  — no action-index prefix')
print('  Task 5 writes: "0:cost:prob |action" — with 0: prefix')
print('  FIX: r"^\\s*(?:0:)?cost:prob \\|action" — optional prefix, accepts both')
print()
print('BUG 4: --cb_adf vs --cb_explore_adf')
print('  OLD: --cb_adf  ← offline-only, cb_type dr/mtr sweep has no effect')
print('  FIX: --cb_explore_adf --epsilon 0.0  ← proper cb_type handling')
print()
print('BUG 5: train_loss proxy sampled only 500 blocks')
print('  OLD: costs[:500] — biased estimate for small datasets')
print('  FIX: collect cost from every block during learning loop')
print()
print('PRESERVED from v1:')
print('  - Quadratic interactions: -q eu -q ea -q ua')
print('  - --clip_p 0.02 in VW args')
print('  - --power_t 0 for fixed learning rate')
print('  - S3 paths, split logic, sweep grid, gate thresholds, alignment parquet')
